# Analyzing DHS microdata

Nigeria 2018 - J:\DATA\DHS_PROG_DHS\NGA\2018
India 2015-2016 - J:\DATA\DHS_PROG_DHS\IND\2015_2016

More recent is available, but probably weird due to COVID

## Documentation

DHS 7 recode manual for variable definitions: https://www.dhsprogram.com/pubs/pdf/DHSG4/Recode7_DHS_10Sep2018_DHSG4.pdf.
However, it does not state which variables are in which file.
The `.MAP` files alongside each data file list the variables in it and what they mean.

Hemoglobin variables of interest: 
- HA53: Hemoglobin level in g/dl with 1 implied dcimal
- HA54: Currently pregnant
- HA55 Result of Hemoglobin measuring.
- HA56 Hemoglobin level adjusted by altitude in g/dl with 1 implied decimal. 

Wealth index variables of interest: 
- HV270: The wealth index is a composite measure of a household's cumulative living standard.
The wealth index is calculated using easy-to-collect data on a household’s ownership of
selected assets, such as televisions and bicycles; materials used for housing construction; and
types of water access and sanitation facilities.
Generated with a statistical procedure known as principal components analysis, the wealth
index places individual households on a continuous scale of relative wealth. DHS separates
all interviewed households into five wealth quintiles to compare the influence of wealth on
various population, health and nutrition indicators. The wealth index is presented in the DHS
Final Reports and survey datasets as a background characteristic
- HV271: Wealth index factor score (5 decimals) 

Pregnancy variables of interest: 
- HML18: Pregnancy status from individual questionnaire. For complete woman’s interviews this is
taken from V213. For incomplete woman's interview with anemia testing the pregnancy
status is taken from this section.
BASE: Women with a completed individual questionnaire or when available information
from the anemia testing section.

List of datasets: https://www.dhsprogram.com/data/dataset/Nigeria_Standard-DHS_2018.cfm?flag=1

Instructions on how to calculate everything can be found at: https://www.dhsprogram.com/pubs/pdf/DHSG1/Guide_to_DHS_Statistics_DHS-7_v2.pdf

In [ ]:
import pandas as pd, numpy as np

%load_ext autoreload
%autoreload 2

!date

In [ ]:
location = "india"

## Load data, name columns

In [ ]:
directory = {
    "india": "/snfs1/DATA/DHS_PROG_DHS/IND/2015_2016/",
    "nigeria": "/snfs1/DATA/DHS_PROG_DHS/NGA/2018/",
    "ethiopia": "/snfs1/DATA/DHS_PROG_DHS/ETH/2016/",
}[location]

### WRA

In [ ]:
%%time

wra_data_file_name = {
    "india": "IND_DHS7_2015_2016_WN_IAIR74FL_Y2018M12D06.DTA",
    "nigeria": "NGA_DHS7_2018_WN_NGIR7AFL_Y2019M11D05.DTA",
    "ethiopia": "ETH_DHS7_2016_WN_ETIR71FL_Y2019M12D11.DTA",
}[location]

wra_columns = {
    "v001": "cluster_number",
    "v002": "household_number",
    "v003": "line_number",
    "v005": "weight",
    "v008": "interview_date",
    "v011": "date_of_birth",
    "v190": "wealth_quintile",
    "v213": "currently_pregnant",
}
wra_data = pd.read_stata(
    directory + wra_data_file_name,
    columns=wra_columns.keys(),
)

In [ ]:
wra_data = wra_data[wra_columns.keys()].rename(columns=wra_columns)

In [ ]:
def recode_wealth_quintile(df):
    return df.map(
        {
            "poorest": "lowest",
            "poorer": "second",
            "middle": "middle",
            "richer": "fourth",
            "richest": "highest",
        }
    )

In [ ]:
wra_data["wealth_quintile"] = recode_wealth_quintile(wra_data.wealth_quintile)

In [ ]:
wra_data["pregnant"] = wra_data.currently_pregnant.str.strip().map({
    "not pregnant, don't know": "not_pregnant",
    "no or unsure": "not_pregnant",
    "pregnant": "pregnant",
    "yes": "pregnant",
})

In [ ]:
wra_data["weight"] = wra_data.weight / 1_000_000

### Births

In [ ]:
birth_data_file_name = {
    "india": "IND_DHS7_2015_2016_BR_IABR74FL_Y2018M12D06.DTA",
    "nigeria": "NGA_DHS7_2018_BR_NGBR7AFL_Y2019M11D05.DTA",
    "ethiopia": "ETH_DHS7_2016_BR_ETBR71FL_Y2019M12D11.DTA",
}[location]

birth_columns = {
    "v005": "weight",
    "v008": "interview_date",
    "v190": "wealth_quintile",
    "b3": "birth_date",
    "m18": "size_of_child",
    "m19": "birth_weight_kilograms",
}
if location == "india":
    birth_columns["s220a"] = "duration_of_pregnancy"
else:
    birth_columns["b20"] = "duration_of_pregnancy"

birth_data = pd.read_stata(
    directory + birth_data_file_name,
    columns=birth_columns.keys(),
)
birth_data

In [ ]:
birth_data = birth_data[birth_columns.keys()].rename(columns=birth_columns)
birth_data["wealth_quintile"] = recode_wealth_quintile(birth_data.wealth_quintile)
birth_data["weight"] = birth_data.weight / 1_000_000
birth_data

### Household members

In [ ]:
%%time

hhm_data_file_name = {
    "india": "IND_DHS7_2015_2016_HHM_IAPR74FL_Y2018M12D06.DTA",
    "nigeria": "NGA_DHS7_2018_HHM_NGPR7AFL_Y2019M11D05.DTA",
    "ethiopia": "ETH_DHS7_2016_HHM_ETPR71FL_Y2019M12D11.DTA",
}[location]

hhm_columns = {
    "hv001": "cluster_number",
    "hv002": "household_number",
    "hv005": "weight",
    "hv008": "date_of_interview",
    "hvidx": "line_number",
    "hv105": "age",
    "hv104": "sex",
    "ha0": "index_to_household",
    "ha1": "age_hemoglobin",
    "hv270": "wealth_quintile",
    "ha53": "hemoglobin_raw_adult",
    "ha56": "hemoglobin_adjusted_adult",
    "ha57": "anemia_adult",
    "hc53": "hemoglobin_raw_child",
    "hc56": "hemoglobin_adjusted_child",
    "hc57": "anemia_child", 
}
hhm_data = pd.read_stata(
    directory + hhm_data_file_name,
    columns=hhm_columns.keys(),
)
hhm_data

In [ ]:
hhm_data = hhm_data[hhm_columns.keys()].rename(columns=hhm_columns)

In [ ]:
id_columns = ["cluster_number", "household_number", "line_number"]
hhm_data = hhm_data.merge(
    wra_data[id_columns + ["pregnant"]],
    on=id_columns,
    how="left",
)
hhm_data["pregnant"] = hhm_data.pregnant.fillna("not_pregnant")

In [ ]:
hhm_data["age"] = hhm_data.age.replace({"95+": 95, "don't know": np.nan}).astype(float)

In [ ]:
hhm_data["sex"] = hhm_data.sex.str.title()

In [ ]:
# Interesting -- sometimes age is quite off between hemoglobin and base.
hhm_data.loc[(hhm_data.age - hhm_data.age_hemoglobin).sort_values().index]

In [ ]:
(hhm_data.age - hhm_data.age_hemoglobin).describe()

In [ ]:
age_bin_edges = [0, 5, 15, 30, 50, 125]
age_group = pd.IntervalIndex(
    pd.cut(hhm_data.age, age_bin_edges, right=False, include_lowest=True)
)
hhm_data["age_start"] = age_group.left
hhm_data["age_end"] = age_group.right

In [ ]:
hhm_data["wealth_quintile"] = recode_wealth_quintile(hhm_data.wealth_quintile)
hhm_data["weight"] = hhm_data.weight / 1_000_000

In [ ]:
for type in ["child", "adult"]:
    for base_col in ["hemoglobin_raw", "hemoglobin_adjusted"]:
        col = f"{base_col}_{type}"
        hhm_data[col] = (
            hhm_data[col]
            .astype(str)
            .replace(
                {
                    "not tested": np.nan,
                    "not present": np.nan,
                    "refused": np.nan,
                    "other": np.nan,
                }
            )
            .astype(float)
        )

In [ ]:
for base_col in ["hemoglobin_raw", "hemoglobin_adjusted", "anemia"]:
    assert (hhm_data.filter(like=base_col).notnull().sum(axis=1) <= 1).all()
    hhm_data[base_col] = np.nan
    # Could use bfill instead of this loop, but it was incredibly slow for me
    for col in hhm_data.filter(like=base_col).columns:
        hhm_data[base_col] = hhm_data[base_col].fillna(hhm_data[col])

In [ ]:
assert (hhm_data[(hhm_data.sex == "male") & (hhm_data.age > 5)].hemoglobin_raw.isnull().all())

## Hemoglobin

In [ ]:
other_overlapping_columns = (
    (set(wra_data.columns) & set(hhm_data.columns)) - set(id_columns) - {"weight"}
)
other_overlapping_columns

In [ ]:
wra_hhm_joined = wra_data.merge(
    hhm_data.drop(columns=["weight"]),
    on=id_columns,
    suffixes=("_wra", "_hhm"),
    how="left",
)
wra_hhm_joined

In [ ]:
for col in other_overlapping_columns:
    assert (wra_hhm_joined[f"{col}_wra"] == wra_hhm_joined[f"{col}_hhm"]).all()
    wra_hhm_joined[col] = wra_hhm_joined[f"{col}_wra"]
    wra_hhm_joined = wra_hhm_joined.drop(columns=[f"{col}_wra", f"{col}_hhm"])

In [ ]:
# https://stackoverflow.com/a/2415343/ with some tweaks
def weighted_avg_and_std(values, weights):
    """
    Return the weighted average and standard deviation.

    They weights are in effect first normalized so that they
    sum to 1 (and so they must not all be 0).

    values, weights -- NumPy ndarrays with the same shape.
    """
    is_nan = np.isnan(values)
    values = values[~is_nan]
    weights = weights[~is_nan]
    average = np.average(values, weights=weights)
    # Fast and numerically precise:
    variance = np.average((values - average) ** 2, weights=weights)
    return pd.Series(
        {
            "mean": average,
            "sd": np.sqrt(variance),
            # https://ngreifer.github.io/WeightIt/reference/ESS.html
            "effective_sample_size": (weights.sum() ** 2) / (weights**2).sum(),
        }
    )

In [ ]:
pregnant_with_anemia_status = wra_hhm_joined[(wra_hhm_joined.pregnant == 'pregnant') & wra_hhm_joined.anemia.notnull()]

In [ ]:
# Matches table 10.21.1
pregnant_with_anemia_status.weight.sum()

In [ ]:
# Within rounding error of table 10.21.1 value
weighted_avg_and_std(
    pregnant_with_anemia_status.anemia == "severe",
    pregnant_with_anemia_status.weight,
)

In [ ]:
# Within rounding error of table 10.21.1 value for any anemia
weighted_avg_and_std(
    pregnant_with_anemia_status.anemia.isin(
        ["severe", "moderate", "mild"]
    ),
    pregnant_with_anemia_status.weight,
)

In [ ]:
assert (
    (pregnant_with_anemia_status.anemia == "severe")
    == (pregnant_with_anemia_status.hemoglobin_adjusted < 70)
).all()

In [ ]:
assert (
    (
        pregnant_with_anemia_status.anemia.isin(
            ["severe", "moderate", "mild"]
        )
    )
    == (pregnant_with_anemia_status.hemoglobin_adjusted < 110)
).all()

In [ ]:
adult_hemoglobin_disparities = (
    wra_hhm_joined.groupby(["wealth_quintile", "pregnant"])
    .apply(lambda df: weighted_avg_and_std(df.hemoglobin_adjusted, weights=df.weight))
    .sort_index()
)
adult_hemoglobin_disparities

In [ ]:
adult_hemoglobin_disparities = adult_hemoglobin_disparities.reset_index()
adult_hemoglobin_disparities = pd.concat([
    adult_hemoglobin_disparities.assign(sex="Female", age_start=15, age_end=125),
    # Assumption: males are like non-pregnant WRA
    adult_hemoglobin_disparities[adult_hemoglobin_disparities.pregnant == "not_pregnant"].assign(sex="Male", age_start=15, age_end=125),
])
adult_hemoglobin_disparities = adult_hemoglobin_disparities.set_index(["sex", "age_start", "age_end", "pregnant", "wealth_quintile"])
adult_hemoglobin_disparities

In [ ]:
child_hemoglobin_disparities = (
    hhm_data[(hhm_data.age <= 5)].assign(age_start=0, age_end=5, pregnant="not_pregnant").groupby(["sex", "age_start", "age_end", "pregnant", "wealth_quintile"])
    .apply(lambda df: weighted_avg_and_std(df.hemoglobin_adjusted, weights=df.weight))
    .sort_index()
)
child_hemoglobin_disparities

In [ ]:
adolescent_hemoglobin_disparities = pd.DataFrame(columns=child_hemoglobin_disparities.columns, index=child_hemoglobin_disparities.index).droplevel(["age_start", "age_end"])
for group in adolescent_hemoglobin_disparities.index:
    child_values = child_hemoglobin_disparities.droplevel(["age_start", "age_end"]).loc[group]
    adult_values = adult_hemoglobin_disparities.droplevel(["age_start", "age_end"]).loc[group]
    adolescent_hemoglobin_disparities.loc[group] = (child_values * 0.5 + adult_values * 0.5).T


In [ ]:
assert adolescent_hemoglobin_disparities.notnull().all().all()
adolescent_hemoglobin_disparities = adolescent_hemoglobin_disparities.reset_index().assign(age_start=5, age_end=15).set_index(list(child_hemoglobin_disparities.index.names))

In [ ]:
hemoglobin_disparities = pd.concat([
    child_hemoglobin_disparities,
    adolescent_hemoglobin_disparities,
    adult_hemoglobin_disparities,
])
hemoglobin_disparities

In [ ]:
results_dir = (
    "../results"
)

In [ ]:
hemoglobin_disparities["mean"].rename("value").to_csv(
    f"{results_dir}/hemoglobin/mean_disparities/{location}.csv"
)

In [ ]:
hemoglobin_disparities["sd"].rename("value").to_csv(
    f"{results_dir}/hemoglobin/sd_disparities/{location}.csv"
)

## Wealth quintile probabilities

Intuitively, you might think these would be equal; but we are looking at subpopulations which can skew.

In [ ]:
group_variables = ["sex", "age_start", "age_end", "pregnant"]

In [ ]:
wealth_quintile_probabilities = (
    hhm_data.groupby(group_variables + ["wealth_quintile"], observed=True).weight.sum() /
    hhm_data.groupby(group_variables, observed=True).weight.sum()
)
assert np.allclose(wealth_quintile_probabilities.groupby(group_variables, observed=True).sum(), 1.0)
wealth_quintile_probabilities

In [ ]:
wealth_quintile_probabilities = wealth_quintile_probabilities.unstack()
wealth_quintile_probabilities

In [ ]:
wealth_quintile_probabilities.to_csv(
    f"{results_dir}/wealth_quintile_probabilities/{location}.csv",
)

## LBWSG

### Birth weight

In [ ]:
birth_data["birth_weight_kilograms"] = birth_data.birth_weight_kilograms.replace(
    {"not weighed at birth": np.nan, "don't know": np.nan}
).astype(float)

In [ ]:
weighted_avg_and_std(birth_data.birth_weight_kilograms, birth_data.weight)

In [ ]:
birth_weight_disparities = birth_data.groupby("wealth_quintile").apply(
    lambda df: weighted_avg_and_std(df.birth_weight_kilograms, df.weight)
)
birth_weight_disparities

In [ ]:
birth_weight_disparities = (
    birth_weight_disparities["mean"].rename("value").reset_index()
)
birth_weight_disparities

In [ ]:
birth_weight_disparities.to_csv(
    f"{results_dir}/birth_weight_disparities/{location}.csv", index=False
)

### Short gestation

All we have here is a self-reported duration of pregnancy.

In [ ]:
# Basically no difference in mean
birth_data.groupby("wealth_quintile").apply(
    lambda df: weighted_avg_and_std(df.duration_of_pregnancy, df.weight)
)

In [ ]:
birth_data["short_gestation"] = np.where(
    birth_data.duration_of_pregnancy.isnull(),
    np.nan,
    birth_data.duration_of_pregnancy < 9.0,
)

In [ ]:
weighted_avg_and_std(birth_data.short_gestation, birth_data.weight)

In [ ]:
birth_data.groupby("wealth_quintile").apply(
    lambda df: weighted_avg_and_std(df.short_gestation, df.weight)
)

The trends in short gestation don't make intuitive sense (?), so we do not plan to use them in the sim.